# Split And SplitManager Exploration

This notebook demonstrates the train/validation/test splitting utilities in `liesel.optim.split`. It starts with a single observation size, then moves to multi-branch models with different observation sizes.

In [1]:
import jax
import jax.numpy as jnp
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.model as lsl
import liesel.optim as opt
from liesel.optim.types import Position

## A Single Split

`Split` owns the row indices. Calling `split_position()` applies those indices to one or more position entries and returns a `PositionSplit`.

In [2]:
position = Position({"x": jnp.arange(10.0)})

splitter = opt.Split(
    position_keys=["x"],
    axis_size=10,
    validate_axis_size=2,
    test_axis_size=1,
    shuffle=False,
)
data = splitter.split_position(position)

{
    "splitter": splitter,
    "indices_train": splitter.indices_train.tolist(),
    "indices_validate": splitter.indices_validate.tolist(),
    "indices_test": splitter.indices_test.tolist(),
    "train": data.train["x"].tolist(),
    "validate": data.validate["x"].tolist(),
    "test": data.test["x"].tolist(),
}

{'splitter': Split(train=7, validate=2, test=1),
 'indices_train': [0, 1, 2, 3, 4, 5, 6],
 'indices_validate': [7, 8],
 'indices_test': [9],
 'train': [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0],
 'validate': [7.0, 8.0],
 'test': [9.0]}

`PositionSplit` exposes the sizes, shares, and validation/test likelihood scaling for scalar split setups.

In [3]:
{
    "axis_size": data.axis_size,
    "train_axis_size": data.train_axis_size,
    "validate_axis_size": data.validate_axis_size,
    "test_axis_size": data.test_axis_size,
    "validate_axis_share": data.validate_axis_share,
    "test_axis_share": data.test_axis_share,
    "validate_sample_scale": data.validate_sample_scale,
}

{'axis_size': 10,
 'train_axis_size': 7,
 'validate_axis_size': 2,
 'test_axis_size': 1,
 'validate_axis_share': 0.2,
 'test_axis_share': 0.1,
 'validate_sample_scale': 3.5}

## Shares And Shuffling

`Split.from_axis_shares()` derives integer split sizes from proportions. With `shuffle=True` and a seed, the generated index order is reproducible.

In [4]:
shuffled_1 = opt.Split.from_axis_shares(
    ["x"], axis_size=10, validate_axis_share=0.2, test_axis_share=0.1, shuffle=True, seed=7
)
shuffled_2 = opt.Split.from_axis_shares(
    ["x"], axis_size=10, validate_axis_share=0.2, test_axis_share=0.1, shuffle=True, seed=7
)

{
    "sizes": (shuffled_1.train_axis_size, shuffled_1.validate_axis_size, shuffled_1.test_axis_size),
    "indices": shuffled_1.indices.tolist(),
    "same_seed_same_indices": bool(jnp.all(shuffled_1.indices == shuffled_2.indices)),
}

{'sizes': (7, 2, 1),
 'indices': [9, 4, 2, 3, 5, 0, 7, 6, 1, 8],
 'same_seed_same_indices': True}

## Splitting Along Different Axes

Each position key can have its own split axis. This is useful when observations are columns for one array and rows for another.

In [5]:
wide_position = Position(
    {
        "x": jnp.arange(12).reshape(3, 4),
        "y": jnp.arange(20).reshape(4, 5),
    }
)

axis_splitter = opt.Split(
    ["x", "y"],
    axis_size=4,
    validate_axis_size=1,
    split_axes={"x": 1, "y": 0},
)
axis_data = axis_splitter.split_position(wide_position)

{
    "train_shapes": {key: value.shape for key, value in axis_data.train.items()},
    "validate_shapes": {key: value.shape for key, value in axis_data.validate.items()},
    "x_validate": axis_data.validate["x"].tolist(),
    "y_validate": axis_data.validate["y"].tolist(),
}

{'train_shapes': {'x': (3, 3), 'y': (3, 5)},
 'validate_shapes': {'x': (3, 1), 'y': (1, 5)},
 'x_validate': [[3], [7], [11]],
 'y_validate': [[15, 16, 17, 18, 19]]}

## Model Convenience For One Observation Size

`PositionSplit.from_model()` can infer the observed position keys and axis size for a regular single-size model.

In [6]:
z = lsl.Var.new_obs(
    jnp.linspace(-1.0, 1.0, 20),
    lsl.Dist(tfd.Normal, loc=0.0, scale=1.0),
    name="z",
)
single_size_model = lsl.Model([z])

single_size_data = opt.PositionSplit.from_model(
    single_size_model,
    validate_axis_share=0.2,
    test_axis_share=0.1,
    shuffle=True,
    seed=11,
)

{
    "type": type(single_size_data).__name__,
    "position_keys": single_size_data.position_keys,
    "sizes": (
        single_size_data.train_axis_size,
        single_size_data.validate_axis_size,
        single_size_data.test_axis_size,
    ),
    "validate_shape": single_size_data.validate["z"].shape,
}

{'type': 'PositionSplit',
 'position_keys': ['z'],
 'sizes': (14, 4, 2),
 'validate_shape': (4,)}

## Multi-Branch Splitting

`SplitManager` coordinates several `Split` objects. Each child owns one branch, so branches can have different observation sizes.

In [7]:
x_values = 1.0 + 0.5 * jax.random.normal(jax.random.key(1), (30,))
y_values = -2.0 + 1.2 * jax.random.normal(jax.random.key(2), (12,))

x_obs = lsl.Var.new_obs(
    x_values,
    lsl.Dist(tfd.Normal, loc=0.0, scale=1.0),
    name="x_obs",
)
y_obs = lsl.Var.new_obs(
    y_values,
    lsl.Dist(tfd.Normal, loc=0.0, scale=1.0),
    name="y_obs",
)
multi_size_model = lsl.Model([x_obs, y_obs])
full_position = multi_size_model.extract_position(["x_obs", "y_obs"])

manual_manager = opt.SplitManager(
    [
        opt.Split(["x_obs"], axis_size=x_values.size, validate_axis_size=6, test_axis_size=3),
        opt.Split(["y_obs"], axis_size=y_values.size, validate_axis_size=2, test_axis_size=1),
    ]
)
manual_data = manual_manager.split_position(full_position)

{
    "position_keys": manual_data.position_keys,
    "axis_sizes": manual_data.axis_sizes,
    "train_axis_sizes": manual_data.train_axis_sizes,
    "validate_axis_sizes": manual_data.validate_axis_sizes,
    "test_axis_sizes": manual_data.test_axis_sizes,
    "train_shapes": {key: value.shape for key, value in manual_data.train.items()},
}

{'position_keys': ['x_obs', 'y_obs'],
 'axis_sizes': (30, 12),
 'train_axis_sizes': (21, 9),
 'validate_axis_sizes': (6, 2),
 'test_axis_sizes': (3, 1),
 'train_shapes': {'x_obs': (21,), 'y_obs': (9,)}}

The scalar aliases are available only when all child values agree. Branch-specific plural properties work for all manager splits.

In [8]:
try:
    manual_data.train_axis_size
except ValueError as error:
    train_axis_size_error = str(error)

{
    "train_axis_sizes": manual_data.train_axis_sizes,
    "train_axis_size_alias_error": train_axis_size_error,
}

{'train_axis_sizes': (21, 9),
 'train_axis_size_alias_error': 'PositionSplitManager.train_axis_size is only available when all contained PositionSplit objects have the same train_axis_size. Use PositionSplitManager.train_axis_sizes for branch-specific values.'}

## Automatic Grouping For Multiple Sizes

`SplitManager.from_model()` groups observed variables by inferred axis size. `PositionSplit.from_model(..., multi_size="manager")` returns a `PositionSplitManager` when multiple sizes are present.

In [9]:
grouped_manager = opt.SplitManager.from_model(
    multi_size_model,
    position_keys=["x_obs", "y_obs"],
    validate_axis_share=0.2,
    test_axis_share=0.1,
    shuffle=True,
    seed=5,
)
grouped_data = grouped_manager.split_position(full_position)

managed_data = opt.PositionSplit.from_model(
    multi_size_model,
    position_keys=["x_obs", "y_obs"],
    validate_axis_share=0.2,
    test_axis_share=0.1,
    shuffle=True,
    seed=5,
    multi_size="manager",
)

{
    "manager_type": type(grouped_manager).__name__,
    "split_type": type(managed_data).__name__,
    "manager_axis_sizes": grouped_manager.axis_sizes,
    "position_split_axis_sizes": managed_data.axis_sizes,
    "position_split_validate_axis_sizes": managed_data.validate_axis_sizes,
}

{'manager_type': 'SplitManager',
 'split_type': 'PositionSplitManager',
 'manager_axis_sizes': (30, 12),
 'position_split_axis_sizes': (30, 12),
 'position_split_validate_axis_sizes': (6, 2)}

## Per-Branch Validation Scaling

For a `PositionSplitManager`, validation likelihood scaling is computed separately for each branch. This keeps branch sizes and validation sizes explicit.

In [10]:
validation_state = multi_size_model.update_state(
    managed_data.validate, multi_size_model.state
)

manual_scaled_log_lik = (
    managed_data.splits[0].validate_sample_scale
    * validation_state[x_obs.dist_node.name].value.sum()
    + managed_data.splits[1].validate_sample_scale
    * validation_state[y_obs.dist_node.name].value.sum()
)

{
    "scaled_log_lik": managed_data.scaled_log_lik(multi_size_model, validation_state),
    "manual_scaled_log_lik": manual_scaled_log_lik,
    "matches_manual": bool(
        jnp.allclose(
            managed_data.scaled_log_lik(multi_size_model, validation_state),
            manual_scaled_log_lik,
        )
    ),
}

{'scaled_log_lik': Array(-74.69591, dtype=float32),
 'manual_scaled_log_lik': Array(-74.69591, dtype=float32),
 'matches_manual': True}

## Helpful Errors

The default scalar constructor rejects multi-size observed data. The manager constructors also reject duplicate position keys and axis-share settings that create validation data for only some branches.

In [11]:
errors = {}

try:
    opt.PositionSplit.from_model(multi_size_model, position_keys=["x_obs", "y_obs"])
except ValueError as error:
    errors["multi_size_default"] = str(error)

try:
    opt.Split(["x_obs", "x_obs"], axis_size=10)
except ValueError as error:
    errors["duplicate_position_keys"] = str(error)

try:
    opt.SplitManager.from_model(
        multi_size_model,
        position_keys=["x_obs", "y_obs"],
        validate_axis_share=0.05,
    )
except ValueError as error:
    errors["mixed_validation_availability"] = str(error)

errors

{'multi_size_default': "PositionSplit.from_model() found observed variables with different axis sizes: {30: ['x_obs'], 12: ['y_obs']}. Use PositionSplit.from_model(..., multi_size='manager') or PositionSplitManager.from_model(...).",
 'duplicate_position_keys': "Duplicate position_keys are not allowed: ['x_obs', 'x_obs']",
 'mixed_validation_availability': 'validate_axis_share=0.05 produced zero validation observations for axis sizes [12], while other split groups received validation data. Increase validate_axis_share, set validate_axis_share=0.0, or construct SplitManager manually.'}

Useful knobs to change while exploring: `validate_axis_share`, `test_axis_share`, `shuffle`, `seed`, per-key `split_axes`, and whether to use `PositionSplit.from_model()` or the explicit `SplitManager` path.